In [1]:
## 1. Reset stage

import shutil
from pathlib import Path

stage_root = Path("/home/jovyan/data/stage/handover_events")

if stage_root.exists():
    shutil.rmtree(stage_root)

stage_root.mkdir(parents=True, exist_ok=True)

print("Stage folder reset")

Stage folder reset


In [2]:
## 2. Config

import csv
import gzip
from pathlib import Path
from datetime import datetime
import pandas as pd

RAW_DIR = Path("/home/jovyan/data/sim/raw/sim_test_decoded/2025/")
STAGE_DIR = Path("/home/jovyan/data/stage/handover_events")
LOG_DIR = Path("/home/jovyan/data/logs")

LOG_DIR.mkdir(parents=True, exist_ok=True)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

# Column indexes (based on your sample)
IDX_EVENT_TS   = 3   # collection_time
IDX_VEHICLE_ID = 5   # sim_id
IDX_CELL_ID    = 18  # cell_id
IDX_IMSI       = 13  # optional, good to keep for identity
IDX_RAT        = 23  # optional, 4G/5G

BATCH_SIZE = 100_000

In [3]:
#Environment Check

import os
import psutil

print("CPU count:", os.cpu_count())
print("RAM GB visible to container:", round(psutil.virtual_memory().total / (1024**3), 2))

CPU count: 24
RAM GB visible to container: 38.98


In [4]:
## Helper functions

def list_input_files(raw_dir: Path):
    exts = {".gz", ".csv", ".txt"}
    return sorted([p for p in raw_dir.rglob("*") if p.is_file() and p.suffix.lower() in exts])

def open_text_file(path: Path):
    if path.suffix.lower() == ".gz":
        return gzip.open(path, mode="rt", newline="", encoding="utf-8", errors="replace")
    return open(path, mode="rt", newline="", encoding="utf-8", errors="replace")

def safe_get(row, idx):
    if idx >= len(row):
        return None
    value = row[idx].strip()
    return value if value != "" else None

def parse_timestamp(value):
    if not value:
        return None
    return pd.to_datetime(value, errors="coerce")

In [5]:
## Batch writer

def flush_batch(rows):
    if not rows:
        return

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["vehicle_id", "event_ts", "cell_id"])

    if df.empty:
        return

    df["event_date"] = df["event_ts"].dt.strftime("%Y-%m-%d")

    for event_date, part in df.groupby("event_date"):
        out_dir = STAGE_DIR / f"event_date={event_date}"
        out_dir.mkdir(parents=True, exist_ok=True)

        ts = datetime.utcnow().strftime("%Y%m%d%H%M%S%f")
        out_file = out_dir / f"stage_{ts}.parquet"
        part.to_parquet(out_file, index=False)

    del df

In [6]:
## File processing
def process_file(path: Path, max_rows=None):
    rows = []
    good = 0
    bad = 0

    with open_text_file(path) as f:
        reader = csv.reader(f)

        for i, row in enumerate(reader, start=1):
            if max_rows and i >= max_rows:
                break

            try:
                event_ts = parse_timestamp(safe_get(row, IDX_EVENT_TS))
                vehicle_id = safe_get(row, IDX_VEHICLE_ID)
                imsi = safe_get(row, IDX_IMSI)
                cell_id = safe_get(row, IDX_CELL_ID)
                rat = safe_get(row, IDX_RAT)

                if pd.isna(event_ts) or not vehicle_id or not cell_id:
                    bad += 1
                    continue

                rows.append({
                    "vehicle_id": vehicle_id,
                    "imsi": imsi,
                    "event_ts": event_ts,
                    "cell_id": str(cell_id),
                    "rat": rat,
                    "source_file": str(path),
                })
                good += 1

                if len(rows) >= BATCH_SIZE:
                    flush_batch(rows)
                    rows = []

            except Exception as e:
                bad += 1
                if bad <= 3:
                    print(f"Row {i}: {e}")
                    print(f"  raw row[:10]: {row[:10]}")

    flush_batch(rows)
    rows = []

    print(f"{path.name} → good={good:,} bad={bad:,}")

In [7]:
## 7. loader

import time

start = time.time()
raw_files = list_input_files(RAW_DIR)
print(f"Found {len(raw_files)} files")

for f in raw_files:
    process_file(f)

elapsed = time.time() - start
print(f"Runtime: {elapsed:.1f}s  ({elapsed/60:.1f} min)")


Found 305 files
2025_01_01.gz → good=179 bad=0
2025_01_02.gz → good=179 bad=0
2025_01_03.gz → good=179 bad=0
2025_01_04.gz → good=179 bad=0
2025_01_05.gz → good=179 bad=0
2025_01_06.gz → good=179 bad=0
2025_01_07.gz → good=171 bad=0
2025_01_08.gz → good=132 bad=0
2025_01_09.gz → good=176 bad=0
2025_01_10.gz → good=179 bad=0
2025_01_11.gz → good=179 bad=0
2025_01_12.gz → good=179 bad=0
2025_01_13.gz → good=179 bad=0
2025_01_14.gz → good=179 bad=0
2025_01_15.gz → good=103 bad=0
2025_01_16.gz → good=0 bad=0
2025_01_17.gz → good=584 bad=0
2025_01_18.gz → good=4,045 bad=1
2025_01_19.gz → good=3,981 bad=1
2025_01_20.gz → good=4,085 bad=1
2025_01_21.gz → good=4,023 bad=1
2025_01_22.gz → good=3,903 bad=0
2025_01_23.gz → good=4,088 bad=0
2025_01_24.gz → good=4,107 bad=0
2025_01_25.gz → good=4,099 bad=0
2025_01_26.gz → good=3,992 bad=0
2025_01_27.gz → good=4,055 bad=0
2025_01_28.gz → good=4,088 bad=0
2025_01_29.gz → good=3,993 bad=0
2025_01_30.gz → good=4,103 bad=0
2025_01_31.gz → good=4,090 bad

KeyboardInterrupt: 

In [ ]:
# one-file check

from pathlib import Path
import pandas as pd

stage_files = list(Path(STAGE_DIR).rglob("*.parquet"))
print("Stage files:", len(stage_files))

df = pd.read_parquet(stage_files[0])
df.head()

In [ ]:
## build results table

import duckdb
con = duckdb.connect("/home/jovyan/data/analytics.duckdb")

con.execute(f"""
drop TABLE handover_events
""")

con.execute(f"""
CREATE TABLE handover_events AS
SELECT *
FROM read_parquet('/home/jovyan/data/stage/handover_events/**/*.parquet')
""")

print("CREATED TABLE handover_events", flush=True)

In [ ]:
import duckdb

con = duckdb.connect("/home/jovyan/data/analytics.duckdb")

con.execute("""
SELECT
    min(event_ts) as min_event,
    max(event_ts) as max_event,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT vehicle_id) AS vehicles,
    COUNT(DISTINCT cell_id) AS cells
FROM handover_events
""").df()



In [ ]:
import duckdb

con = duckdb.connect("/home/jovyan/data/analytics.duckdb")

con.execute("""
SELECT *
FROM handover_events
LIMIT 1
""").df()


In [ ]:
# after loading col5 as candidate_id

import duckdb

con = duckdb.connect("/home/jovyan/data/analytics.duckdb")
#con.execute("DESCRIBE handover_events").df()
con.execute("select count(1) from handover_events").df()
